# Week 04. 정적 웹 크롤링과 스크래핑

4주차는 HTML 문서를 가져와 필요한 정보를 선택자로 추출하는 흐름을 다룹니다.
네트워크 요청 대신 로컬 HTML 문자열을 사용해서 항상 오류 없이 실행되게 했습니다.


## 제출자 정보

- 이름: 이성민
- 학과: 소프트웨어융합과
- 학년: 2학년
- 학번: 2151050


## 1. HTML 샘플 준비

실제 웹 페이지도 결국 문자열 형태의 HTML입니다.
`BeautifulSoup`은 이 문자열을 탐색 가능한 구조로 바꿔 줍니다.


In [1]:
from bs4 import BeautifulSoup
import pandas as pd

html = '''
<html>
  <body>
    <article class="book" data-category="python">
      <h3>Clean Python</h3><p class="price">32000</p><p class="rating">5</p>
    </article>
    <article class="book" data-category="data">
      <h3>Data Notebook</h3><p class="price">28000</p><p class="rating">4</p>
    </article>
    <article class="book" data-category="web">
      <h3>Web Scraping Lab</h3><p class="price">25000</p><p class="rating">4</p>
    </article>
  </body>
</html>
'''

soup = BeautifulSoup(html, "html.parser")
print(soup.select_one("article.book h3").get_text(strip=True))


Clean Python


## 2. 선택자로 필요한 데이터 추출

`select()`는 CSS 선택자를 사용합니다.
제목, 가격, 평점처럼 화면에 흩어진 값을 한 행의 데이터로 모읍니다.


In [2]:
rows = []
for card in soup.select("article.book"):
    rows.append(
        {
            "title": card.select_one("h3").get_text(strip=True),
            "category": card["data-category"],
            "price": int(card.select_one(".price").get_text(strip=True)),
            "rating": int(card.select_one(".rating").get_text(strip=True)),
        }
    )

books = pd.DataFrame(rows)
books


,title,category,price,rating
0,Clean Python,python,32000,5
1,Data Notebook,data,28000,4
2,Web Scraping Lab,web,25000,4


## 3. 간단한 텍스트 인코딩

스크래핑한 텍스트는 분석 전에 숫자 형태로 바꾸는 경우가 많습니다.
여기서는 제목에 등장하는 단어를 CountVectorizer로 세어 봅니다.


In [3]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
matrix = vectorizer.fit_transform(books["title"])
word_counts = pd.DataFrame(matrix.toarray(), columns=vectorizer.get_feature_names_out())
word_counts


,clean,data,lab,notebook,python,scraping,web
0,1,0,0,0,1,0,0
1,0,1,0,1,0,0,0
2,0,0,1,0,0,1,1


## 정리

- 정적 크롤링은 HTML 안에 이미 들어 있는 정보를 추출하는 방식입니다.
- CSS 선택자는 필요한 태그를 정확히 찾는 핵심 도구입니다.
- 추출 결과는 DataFrame으로 정리하면 통계와 텍스트 분석으로 이어갈 수 있습니다.
